## PERFORMANCE OPTIMIZATION & COMPARATIVE ANALYSIS

### Objectives
- Build a business KPI
- Compare Liquid Clustering vs Partition + ZORDER
- Benchmark query performance
- Simulate Incremental Loads
- Perform MERGE INTO for Late Arriving Data
- Demonstrate Delta Lake Time Travel

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import time

In [0]:
%run "/Workspace/Users/sanskruti.r.sampate@v4c.ai/vstone/src/notebooks/00_configs"

In [0]:
## LOAD FACT TABLE

fact_df = spark.read.table(FACT_TABLE)

print(f"Total Records : {fact_df.count():,}")

display(fact_df.limit(10))

In [0]:
fact_df.printSchema()

%md
## Business KPI

### Airlines with Highest Cancellation Rate

This KPI helps identify airlines with the highest percentage of cancelled flights.

Business Use Cases:
- Airline performance monitoring
- Operational improvement
- Customer satisfaction analysis
- Root cause investigation

In [0]:
# ============================================================
# AIRLINE CANCELLATION KPI
# ============================================================

airline_df = spark.read.table(AIRLINE_DIM)

kpi_df = (

    fact_df

    .groupBy("airline_sk")

    .agg(

        F.count("*").alias("total_flights"),

        F.sum(
            F.when(
                F.col("cancelled") == True,
                1
            ).otherwise(0)
        ).alias("cancelled_flights")

    )

    .withColumn(

        "cancellation_rate",

        F.round(

            F.col("cancelled_flights")
            /
            F.col("total_flights")
            * 100,

            2

        )

    )

    .join(
        airline_df,
        "airline_sk"
    )

    .orderBy(
        F.desc("cancellation_rate")
    )

)

display(kpi_df)


## 2. CREATE PERFORMANCE COMPARISON TABLES

We will create two identical copies of the Gold Fact table.

Table 1
- Liquid Clustering

Table 2
- Traditional Partition + ZORDER

These tables will be benchmarked using identical analytical queries.

In [0]:
LIQUID_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.fact_flight_delays_liquid"

ZORDER_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.fact_flight_delays_zorder"

print(LIQUID_TABLE)
print(ZORDER_TABLE)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {LIQUID_TABLE}")

spark.sql(f"DROP TABLE IF EXISTS {ZORDER_TABLE}")

print("Old comparison tables removed.")

In [0]:
spark.sql(f"""

CREATE TABLE {LIQUID_TABLE}

CLUSTER BY (
    airline_sk,
    flight_date_key
)

AS

SELECT *

FROM {FACT_TABLE}

""")

print("Liquid Clustered table created.")

In [0]:
spark.sql(f"""
CREATE TABLE {ZORDER_TABLE}

USING DELTA

PARTITIONED BY (flight_date_key)

AS

SELECT *

FROM {FACT_TABLE}

""")

print("Partitioned table created.")

In [0]:
spark.sql(f"""
OPTIMIZE {ZORDER_TABLE}
ZORDER BY (
    airline_sk,
    origin_airport_sk,
    dest_airport_sk
)
""")

print("ZORDER Optimization completed.")



## 3. PERFORMANCE BENCHMARKING


We will execute the same analytical query on:

1. Liquid Clustered Table
2. Partition + ZORDER Table

Then compare execution times.

In [0]:
# ============================================================
# BENCHMARK QUERY
# ============================================================

BENCHMARK_QUERY = """

SELECT

    airline_sk,

    AVG(depdelayminutes) AS avg_departure_delay,

    COUNT(*) AS total_flights

FROM {table}

WHERE flight_date_key BETWEEN 20190101 AND 20191231

GROUP BY airline_sk

ORDER BY avg_departure_delay DESC

"""

In [0]:
# ============================================================
# BENCHMARK FUNCTION
# ============================================================

import time

def benchmark(table_name, label):

    query = BENCHMARK_QUERY.format(table=table_name)

    start = time.time()

    spark.sql(query).collect()

    end = time.time()

    elapsed = round(end - start, 2)

    print(f"{label} : {elapsed} seconds")

    return elapsed

In [0]:
# ============================================================
# RUN BENCHMARK
# ============================================================

print("Running Benchmark...\n")

liquid_time = benchmark(
    LIQUID_TABLE,
    "Liquid Clustering"
)

zorder_time = benchmark(
    ZORDER_TABLE,
    "Partition + ZORDER"
)

In [0]:
comparison = spark.createDataFrame(

    [

        ("Liquid Clustering", liquid_time),

        ("Partition + ZORDER", zorder_time)

    ],

    ["Strategy", "Execution_Time_Seconds"]

)

display(comparison)

In [0]:
print("="*50)

if liquid_time < zorder_time:

    print(" Best Strategy : Liquid Clustering")

elif zorder_time < liquid_time:

    print(" Best Strategy : Partition + ZORDER")

else:

    print(" Both performed similarly")

print("="*50)

##  4. SIMULATE INCREMENTAL LOAD

This section simulates a new batch of flight records arriving after the
initial data load.

These records represent incremental data that needs to be processed.

In [0]:
from pyspark.sql import Row
from datetime import datetime

incremental_data = [

    Row(
        flight_sk=99999991,
        airline_sk=1,
        origin_airport_sk=10,
        dest_airport_sk=20,
        flight_date_key=20200115,
        flightdate="2020-01-15",
        depdelayminutes=35.0,
        arrdelayminutes=28.0,
        airtime=120.0,
        actualelapsedtime=145.0,
        distance=900.0,
        cancelled=False,
        diverted=False,
        load_dt=datetime.now(),
        source="Incremental Load"
    ),

    Row(
        flight_sk=99999992,
        airline_sk=2,
        origin_airport_sk=15,
        dest_airport_sk=30,
        flight_date_key=20200116,
        flightdate="2020-01-16",
        depdelayminutes=15.0,
        arrdelayminutes=12.0,
        airtime=90.0,
        actualelapsedtime=110.0,
        distance=650.0,
        cancelled=False,
        diverted=False,
        load_dt=datetime.now(),
        source="Incremental Load"
    )

]

In [0]:
incremental_df = spark.createDataFrame(incremental_data)

display(incremental_df)

In [0]:
incremental_df.createOrReplaceTempView("incremental_flights")

print("Temporary Incremental View Created")

In [0]:
%sql

COMMENT ON TABLE vstone.gold.fact_flight_delays IS
'Gold Fact Table containing flight performance metrics used for enterprise analytics.';